In [29]:
from dotenv import load_dotenv
import anthropic, json, pandas as pd
from pathlib import Path
import os
import re
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
from IPython.display import display

In [30]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

In [31]:
load_dotenv(root / ".env")
assert os.getenv("ANTHROPIC_API_KEY"), "Falta ANTHROPIC_API_KEY en .env"

# Cliente de producción. max_retries=5: el SDK reintenta por su cuenta los 429
# (rate limit), 5xx y errores de conexión con backoff exponencial, respetando la
# cabecera retry-after. Solo si fallan los 6 intentos llega la excepción a nuestro código.
# El cliente es thread-safe: todos los hilos pueden compartir este mismo objeto.
client = anthropic.Anthropic(max_retries=5)

MODELO = "claude-haiku-5-5"

In [32]:
# df = pd.read_csv(root / "csv/ejercicios_clasificados.csv", encoding="utf-8")

In [33]:
# df[(df.asignatura == "Mates CCSS") & (df.tema == "funciones ccss" )]

In [34]:
def parse_response(texto):
    texto = re.sub(r"```(?:json)?", "", texto).strip()
    dec = json.JSONDecoder()
    objs, i = [], 0
    while True:
        i = texto.find("{", i)
        if i == -1:
            break
        try:
            obj, fin = dec.raw_decode(texto, i)
            if isinstance(obj, dict):
                objs.append(obj)
            i = fin
        except json.JSONDecodeError:
            i += 1

    if not objs:
        raise ValueError(f"Sin JSON en la respuesta: {texto[:150]!r}")

    tipos, obs = [], []
    for o in objs:
        for t in str(o.get("tipo_ejercicio", "")).split("|"):
            t = t.strip()
            if t and t not in tipos:
                tipos.append(t)
        for x in str(o.get("observaciones", "") or "").split(";"):
            x = x.strip()
            if x and x not in obs:
                obs.append(x)

    orden = {"baja": 0, "media": 1, "alta": 2}
    conf = min((o.get("confianza", "baja") for o in objs),
               key=lambda c: orden.get(c, 0))
    return {"tipo_ejercicio": "|".join(tipos),
            "observaciones": "; ".join(obs),
            "confianza": conf}

In [35]:
def build_request(enunciado, system_md, modelo=MODELO):
    # Parámetros de la petición en un solo sitio, para que la calibración
    # mida exactamente la misma llamada que la clasificación.
    return dict(
        model=modelo,
        max_tokens=1000,
        system=[{"type": "text", "text": system_md,
                 "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": f"Enunciado:\n{enunciado}"}],
    )


def classify(enunciado, system_md, modelo=MODELO):
    resp = client.messages.create(**build_request(enunciado, system_md, modelo))
    # El modelo devuelve también ThinkingBlocks: nos quedamos solo con el texto
    texto = "".join(b.text for b in resp.content if b.type == "text")
    try:
        return parse_response(texto)
    except Exception as e:
        # Para depurar: guarda la respuesta cruda junto al error
        raise ValueError(f"{e} | RAW: {texto[:300]!r}") from e

In [36]:
def classify_statements(asignatura, tema, prompt_path,
                        csv_enun=root / "csv/ejercicios_clasificados.csv",
                        max_workers=8):

    # cargar dataframes
    df_enun = pd.read_csv(csv_enun, encoding="utf-8")
    # máscara para filtrar asignatura y tema
    mask = (df_enun.asignatura == asignatura) & (df_enun.tema == tema)
    df_to_be_classified = df_enun.loc[mask, ["id", "enunciado"]]

    system_md = Path(prompt_path).read_text(encoding="utf-8")

    # borrar el archivo si existe clasificación previa
    output_file = root / "classifier/response.jsonl"
    output_file.parent.mkdir(parents=True, exist_ok=True)
    output_file.unlink(missing_ok=True)

    n_ok = 0
    errores = Counter()          # nº de errores por tipo
    t0 = time.perf_counter()

    # El pool crea `max_workers` hilos. Cada llamada a la API pasa casi todo
    # su tiempo esperando la respuesta de red, así que mientras un hilo espera
    # los demás pueden lanzar sus peticiones: varias llamadas en vuelo a la vez.
    with ThreadPoolExecutor(max_workers=max_workers) as pool, \
         output_file.open("w", encoding="utf-8") as f:

        # submit() encola la tarea y devuelve al instante un Future (un
        # "resguardo" del resultado que llegará). Guardamos Future -> id para
        # saber a qué enunciado corresponde cada resultado.
        futuros = {pool.submit(classify, row.enunciado, system_md): row.id
                   for row in df_to_be_classified.itertuples(index=False)}

        try:
            # as_completed() va entregando los futuros según TERMINAN, no en el
            # orden en que se enviaron. Este bucle corre en el hilo principal:
            # es el único que escribe en el archivo.
            for fut in tqdm(as_completed(futuros), total=len(futuros),
                            desc=f"{asignatura} · {tema}"):
                id_ = futuros[fut]
                try:
                    # result() devuelve lo que devolvió classify() o relanza
                    # aquí, en el hilo principal, la excepción del hilo de trabajo.
                    r = fut.result()
                    r["id"] = id_
                    n_ok += 1
                except anthropic.AuthenticationError:
                    raise   # clave mala: no tiene sentido seguir (ver except de abajo)
                except anthropic.RateLimitError as e:
                    # 429 que ha sobrevivido a los reintentos del SDK
                    r = {"id": id_, "error": str(e), "error_tipo": "rate_limit"}
                except Exception as e:
                    r = {"id": id_, "error": str(e), "error_tipo": type(e).__name__}

                if "error_tipo" in r:
                    errores[r["error_tipo"]] += 1
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
                f.flush()   # que quede en disco aunque el proceso se corte

        except (anthropic.AuthenticationError, KeyboardInterrupt):
            # Cancela los futuros que aún no han empezado. Los que ya están en
            # marcha no se pueden interrumpir: el `with` espera a que acaben
            # (como mucho max_workers llamadas) antes de salir.
            pool.shutdown(wait=False, cancel_futures=True)
            print(f"Proceso detenido. {n_ok} resultados guardados en {output_file}")
            raise

    dt = time.perf_counter() - t0
    n = len(df_to_be_classified)
    print(f"Tiempo total: {dt:.1f} s ({n / dt:.2f} enunciados/s con {max_workers} hilos)")
    print(f"OK: {n_ok} / {n}")
    for tipo, k in errores.most_common():
        print(f"  error {tipo}: {k}")

    if n == 0:
        print("No hay enunciados para esa asignatura y tema.")
        return pd.DataFrame()

    # leer los ejercicios clasificados desde el json (ruta absoluta)
    df_classified = pd.read_json(output_file, lines=True)

    # as_completed desordena los resultados: los devolvemos al orden del CSV
    orden = {id_: i for i, id_ in enumerate(df_to_be_classified.id)}
    df_classified = (df_classified.sort_values("id", key=lambda s: s.map(orden))
                                  .reset_index(drop=True))

    # añadir campo de asignatura y tema
    df_classified["asignatura"] = asignatura
    df_classified["tema"] = tema

    return df_classified

## Calibración del número de hilos

Prueba varios valores de `max_workers` con una muestra pequeña del tema y recomienda el mayor que no provoque errores 429.
No escribe en `classifier/response.jsonl`. Usa un cliente con `max_retries=0` para que los 429 no queden ocultos tras los reintentos del SDK.

In [37]:
# Cliente solo para calibrar: sin reintentos, así cada 429 llega tal cual.
client_cal = anthropic.Anthropic(max_retries=0)


def _llamada_calibracion(enunciado, system_md, modelo):
    """Misma petición que classify(), pero devuelve latencia, tokens y cabeceras."""
    t0 = time.perf_counter()
    # with_raw_response da acceso a la respuesta HTTP (cabeceras incluidas);
    # .parse() la convierte en el Message habitual.
    raw = client_cal.messages.with_raw_response.create(
        **build_request(enunciado, system_md, modelo))
    latencia = time.perf_counter() - t0
    return latencia, raw.parse().usage.output_tokens, raw.headers


def calibrar_hilos(asignatura, tema, prompt_path,
                   csv_enun=root / "csv/ejercicios_clasificados.csv",
                   n_muestra=24, valores=(1, 4, 8, 12, 16), modelo=MODELO):
    df_enun = pd.read_csv(csv_enun, encoding="utf-8")
    mask = (df_enun.asignatura == asignatura) & (df_enun.tema == tema)
    df_tema = df_enun.loc[mask, ["id", "enunciado"]]
    n_total = len(df_tema)
    if n_total == 0:
        raise ValueError("No hay enunciados para esa asignatura y tema.")
    muestra = df_tema.sample(min(n_muestra, n_total), random_state=0)
    system_md = Path(prompt_path).read_text(encoding="utf-8")

    # Llamada de calentamiento: escribe el system prompt en la caché (si no,
    # la primera ronda pagaría y tardaría más) y lee los límites de la cuenta.
    _, _, cabeceras = _llamada_calibracion(muestra.enunciado.iloc[0], system_md, modelo)
    limites = {k.removeprefix("anthropic-ratelimit-"): v
               for k, v in cabeceras.items() if k.startswith("anthropic-ratelimit-")}
    print("Límites de la cuenta (cabeceras anthropic-ratelimit-*):")
    display(pd.Series(limites, name="valor").to_frame())

    filas = []
    for w in valores:
        latencias, tokens_out = [], []
        n_429 = n_otros = 0
        t0 = time.perf_counter()
        with ThreadPoolExecutor(max_workers=w) as pool:
            futuros = [pool.submit(_llamada_calibracion, e, system_md, modelo)
                       for e in muestra.enunciado]
            for fut in tqdm(as_completed(futuros), total=len(futuros),
                            desc=f"{w} hilos", leave=False):
                if fut.cancelled():      # cancelado tras un 429: no hay resultado
                    continue
                try:
                    lat, out, _ = fut.result()
                    latencias.append(lat)
                    tokens_out.append(out)
                except anthropic.AuthenticationError:
                    pool.shutdown(wait=False, cancel_futures=True)
                    raise
                except anthropic.RateLimitError:
                    n_429 += 1
                    # En cuanto aparece un 429 dejamos de lanzar llamadas nuevas
                    pool.shutdown(wait=False, cancel_futures=True)
                except Exception:
                    n_otros += 1         # p. ej. 529 overloaded o error de red
        dt = time.perf_counter() - t0

        filas.append({
            "max_workers": w,
            "completadas": len(latencias),
            "tiempo_s": round(dt, 1),
            "enunciados_por_s": round(len(latencias) / dt, 2),
            "latencia_media_s": round(sum(latencias) / len(latencias), 2) if latencias else None,
            "n_429": n_429,
            "otros_errores": n_otros,
            "tokens_salida_medios": round(sum(tokens_out) / len(tokens_out)) if tokens_out else None,
        })
        if n_429:
            print(f"429 con {w} hilos: se detiene la calibración.")
            break

    df_cal = pd.DataFrame(filas)
    display(df_cal)

    sin_429 = df_cal[df_cal.n_429 == 0]
    if sin_429.empty:
        print("Hay 429 incluso con el valor más bajo probado: usa 1 hilo o espera un minuto.")
        return df_cal, None

    mejor = sin_429.loc[sin_429.max_workers.idxmax()]
    recomendado = int(mejor.max_workers)
    est = n_total / mejor.enunciados_por_s
    print(f"Recomendado: max_workers={recomendado} "
          f"({mejor.enunciados_por_s} enunciados/s en la prueba).")
    print(f"Estimación para los {n_total} enunciados de «{tema}»: "
          f"{est:.0f} s (~{est / 60:.1f} min).")
    if recomendado == max(valores):
        print("Ningún valor ha dado 429: podrías probar valores más altos.")
    return df_cal, recomendado

In [38]:
# df_cal, workers_recomendado = calibrar_hilos(asignatura="Química", tema="equilibrio químico",
#                                              prompt_path=root / "classify_using_api/equilibrio_quimico.md",
#                                              csv_enun=root / "classify_using_api/df_equilibrio_quimico_sample.csv",n_muestra=24)

In [39]:
df_classified = classify_statements(asignatura="Química", tema="equilibrio químico",
                                    prompt_path=root / "classify_using_api/equilibrio_quimico.md",
                                    csv_enun=root / "classify_using_api/df_equilibrio_quimico.csv",
                                    max_workers=16)  # o workers_recomendado tras calibrar

Química · equilibrio químico:   0%|          | 0/183 [00:00<?, ?it/s]

Tiempo total: 28.9 s (6.34 enunciados/s con 16 hilos)
OK: 183 / 183


In [40]:
df_classified.confianza.value_counts()

confianza
alta     136
media     46
baja       1
Name: count, dtype: int64

In [41]:
df_classified.head()

,tipo_ejercicio,observaciones,confianza,id,asignatura,tema
0,Equilibrio heterogéneo|Cálculo de Kc y Kp|Ley ...,Equilibrio con C(s) y masa de sólido sin reacc...,alta,quimica_equilibrio_quimico_2026_junio_ejercici...,Química,equilibrio químico
1,Cálculo de Kc y Kp|Grado de disociación (cálculo),,alta,quimica_equilibrio_quimico_2025_julio_ejercici...,Química,equilibrio químico
2,Equilibrio heterogéneo|Cálculo de Kc y Kp,Pide masa de sólido sin reaccionar (equilibrio...,alta,quimica_equilibrio_quimico_2025_junio_ejercici...,Química,equilibrio químico
3,Principio de LeChatelier|Equilibrio heterogéneo,,alta,quimica_equilibrio_quimico_2024_julio_ejercici...,Química,equilibrio químico
4,Cálculo de Kc y Kp|Grado disociación (dato),,alta,quimica_equilibrio_quimico_2024_julio_ejercici...,Química,equilibrio químico


In [42]:
df_classified.isna().sum()

tipo_ejercicio    0
observaciones     0
confianza         0
id                0
asignatura        0
tema              0
dtype: int64

In [45]:
df_classified.tipo_ejercicio = df_classified.tipo_ejercicio.str.split("|")

In [47]:
df_classified = df_classified.explode("tipo_ejercicio", ignore_index=True)

In [48]:
df_classified.tipo_ejercicio.unique()

<ArrowStringArray>
[        'Equilibrio heterogéneo',             'Cálculo de Kc y Kp',
                  'Ley de Dalton', 'Grado de disociación (cálculo)',
       'Principio de LeChatelier',       'Grado disociación (dato)',
  'Equilibrio en función de alfa',           'Cociente de reacción',
                 'sin clasificar']
Length: 9, dtype: str

In [49]:
df_classified.tipo_ejercicio.value_counts()

tipo_ejercicio
Cálculo de Kc y Kp                112
Equilibrio heterogéneo             60
Ley de Dalton                      52
Principio de LeChatelier           50
Grado de disociación (cálculo)     25
Grado disociación (dato)           16
Cociente de reacción                8
Equilibrio en función de alfa       7
sin clasificar                      1
Name: count, dtype: int64

In [50]:
df_classified.to_csv(root / "classify_using_api/equilibrio_quimico_clasificados_id_solo.csv", index = False, encoding="utf-8")